# Herramientas y agentes con Ollama

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion6/1-herramientas-y-agentes-con-ollama.ipynb)

En este notebook vamos a construir el escalón inicial de la nueva unidad: un LLM con acceso a herramientas, pero todavía sin MCP. La idea es entender primero el problema pedagógico y técnico que queremos resolver: cómo pasar de un modelo que solo genera texto a un agente capaz de delegar cálculos exactos y consultas a servicios externos.

### Referencias
- [LangGraph - Prebuilt agents](https://docs.langchain.com/oss/python/langgraph/overview)
- [LangChain Ollama integration](https://docs.langchain.com/oss/python/integrations/providers/ollama)
- [Open-Meteo API](https://open-meteo.com/)
- [Ollama](https://ollama.com/)


In [ ]:
import warnings

warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
!test '{IN_COLAB}' = 'True' && pip install langchain langchain-core langchain-ollama langgraph httpx ollama colab-xterm


### Cargando a Ollama

Seguiremos la misma filosofía de la sesión de RAG: usar un modelo local a través de Ollama. Para esta clase nos interesa que el modelo tenga soporte razonable para *invocación de herramientas*; `llama3.2:3b` suele ser suficiente para una demostración ligera. Si notas respuestas erráticas al invocar herramientas, puedes probar un modelo más fuerte para uso de herramientas, por ejemplo `qwen2.5:3b`.

In [ ]:
!sudo apt install zstd -y
!if ! type ollama > /dev/null; then curl -fsSL https://ollama.com/install.sh | sh; else echo "Ollama ya está instalado."; fi


## Atención

Si trabajas en Colab, recuerda ejecutar Ollama en la terminal embebida para que el servidor quede activo. Si corres el notebook en local con `ollama serve` ya encendido, puedes omitir esa parte.

In [ ]:
%load_ext colabxterm
%xterm


Para este notebook usaremos el modelo `llama3.2:3b`.


In [ ]:
!ollama pull llama3.2:3b


## Primero: ¿qué pasa sin herramientas?

Antes de crear un agente, conviene observar que un LLM por sí solo puede aproximar operaciones o responder desde memoria, pero no tiene por qué ser exacto ni estar conectado con el estado actual del mundo.

In [ ]:
from langchain_ollama import ChatOllama

MODEL = 'llama3.2:3b'
llm = ChatOllama(model=MODEL, temperature=0)

respuesta_sin_tools = llm.invoke(
    'Responde en dos líneas: ¿cuánto es (125 * 17) + 938 y cuál es la temperatura actual en Cali, Colombia?'
)
print(respuesta_sin_tools.content)


Lo normal es que el modelo intente responder algo útil, pero sin garantías. Ese es justamente el punto de partida: si necesitamos precisión aritmética o datos actuales, hace falta exponer capacidades externas de forma controlada.

## Definimos dos herramientas sencillas

Usaremos exactamente los dos casos que reaparecerán en el notebook de MCP: una calculadora segura y una consulta de clima actual usando Open-Meteo. Así podremos comparar la misma tarea antes y después de introducir el protocolo.

In [ ]:
import ast
import operator as op
import httpx
from langchain_core.tools import tool

ALLOWED_BIN_OPS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.Mod: op.mod,
}
ALLOWED_UNARY_OPS = {
    ast.UAdd: op.pos,
    ast.USub: op.neg,
}

def evaluate_expression(expression: str):
    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in ALLOWED_BIN_OPS:
            return ALLOWED_BIN_OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in ALLOWED_UNARY_OPS:
            return ALLOWED_UNARY_OPS[type(node.op)](_eval(node.operand))
        raise ValueError('La expresión contiene operadores no soportados.')

    parsed = ast.parse(expression, mode='eval')
    return _eval(parsed)

@tool
def calculadora(expression: str) -> str:
    """Evalúa una expresión aritmética segura con +, -, *, /, %, ** y paréntesis."""
    result = evaluate_expression(expression)
    return f'Resultado exacto: {result}'

@tool
def clima_actual(city: str) -> str:
    """Consulta el clima actual de una ciudad usando Open-Meteo."""
    geocode_response = httpx.get(
        'https://geocoding-api.open-meteo.com/v1/search',
        params={'name': city, 'count': 1, 'language': 'es', 'format': 'json'},
        timeout=30.0,
    )
    geocode_response.raise_for_status()
    geocode_data = geocode_response.json()
    if not geocode_data.get('results'):
        return f'No encontré información para la ciudad: {city}'

    location = geocode_data['results'][0]
    weather_response = httpx.get(
        'https://api.open-meteo.com/v1/forecast',
        params={
            'latitude': location['latitude'],
            'longitude': location['longitude'],
            'current': 'temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code',
            'timezone': 'auto',
        },
        timeout=30.0,
    )
    weather_response.raise_for_status()
    weather_data = weather_response.json()['current']

    return (
        f"Clima actual en {location['name']}, {location.get('country', '')}: "
        f"temperatura {weather_data['temperature_2m']}°C, "
        f"humedad {weather_data['relative_humidity_2m']}%, "
        f"viento {weather_data['wind_speed_10m']} km/h, "
        f"weather_code {weather_data['weather_code']}."
    )


## Construimos un agente con LangGraph

Ahora sí aparece la capa agentic: el LLM decide cuándo necesita una herramienta, la invoca y luego integra el resultado dentro de su respuesta final.

In [ ]:
from langgraph.prebuilt import create_react_agent

tools = [calculadora, clima_actual]
agent = create_react_agent(model=llm, tools=tools)

def preguntar_al_agente(question: str):
    result = agent.invoke({'messages': [('user', question)]})
    final_message = result['messages'][-1]
    return final_message.content, result


## Caso 1: aritmética verificable


In [ ]:
respuesta_calculo, traza_calculo = preguntar_al_agente(
    '¿Cuánto es (125 * 17) + 938? Responde en español y menciona el resultado final.'
)
print(respuesta_calculo)


Aquí el modelo deja de adivinar: delega el cálculo y luego redacta la respuesta. Este es uno de los beneficios más sencillos de enseñar, porque el contraste con el modo “solo LLM” se ve de inmediato.

## Caso 2: grounding con una API pública


In [ ]:
respuesta_clima, traza_clima = preguntar_al_agente(
    'Consulta el clima actual de Cali, Colombia, y resume la información más importante en una oración.'
)
print(respuesta_clima)


En este segundo ejemplo el valor ya no es la aritmética, sino el acceso a información actual. El agente no “sabe” el clima por sí mismo: sabe cuándo debe preguntar a una herramienta que sí puede consultarlo.

In [ ]:
def resumir_traza(result):
    for message in result['messages']:
        print(type(message).__name__)
        content = getattr(message, 'content', None)
        if content:
            print(content)
        tool_calls = getattr(message, 'tool_calls', None)
        if tool_calls:
            print(tool_calls)
        print('-' * 80)

resumir_traza(traza_calculo)


## Conclusiones

- Un agente con herramientas ya resuelve el problema práctico de delegar tareas precisas.
- Aún así, las herramientas siguen viviendo pegadas a esta aplicación concreta.
- En el siguiente notebook moveremos exactamente estas capacidades a servidores MCP para separar mejor responsabilidades y ganar interoperabilidad.
